# Notebook 05 — Evaluation: Tables and Figures

**Project:** AV-XAI
**Input:** `checkpoints/best.pt` from nb04 (val BAcc 0.7774 at epoch 8)
**Output:** All paper-ready tables and figures from a single test-set inference pass

## What gets generated

### Tables (CSV)
| # | File | Contents |
|---|---|---|
| T1 | `results/tables/main_results.csv` | 4-class: Acc, BAcc, AUC (macro OvR), F1 (macro, weighted) |
| T2 | `results/tables/binary_results.csv` | Real-vs-fake metrics — both via dedicated head and via 4-class derivation |
| T3 | `results/tables/per_class_breakdown.csv` | Precision/Recall/F1/Support per class |
| T4 | `results/tables/ablation_modality.csv` | Video-only / Audio-only / Fused / Ensemble — all metrics |
| T5 | `results/tables/ablation_fusion.csv` | Feature-level / Decision-level / Ensemble fusion comparison |
| T6 | `results/tables/baseline_comparison.csv` | Ours vs BA-TFD / BA-TFD+ (literature numbers to fill manually) |
| T7 | `results/tables/training_log.csv` | Already exists from nb04 — just verified |

### Figures (PNG + PDF)
| # | File | Contents |
|---|---|---|
| F1 | `results/figures/training_curves.png/.pdf` | Train loss + val BAcc over 8 epochs |
| F2 | `results/figures/confusion_matrix_4class.png/.pdf` | 4×4 confusion matrix, row-normalized |
| F3 | `results/figures/confusion_matrix_binary.png/.pdf` | 2×2 binary confusion matrix |
| F4 | `results/figures/roc_curves.png/.pdf` | Per-class ROC + macro average |
| F5 | `results/figures/modality_ablation_bar.png/.pdf` | Bar chart of 4 modality conditions |

## Free wins from the model architecture

The model outputs **four sets of 4-class logits** in a single forward pass:
- `logits_video` — video-only head (uses just the ViViT [CLS] token)
- `logits_audio` — audio-only head (uses just WavLM mean-pool)
- `logits_fused` — feature-level fusion only (the 2-token transformer output)
- `logits_4class` — ensemble (mean of softmax across the three above)

This means the **entire modality ablation table** (T4) and **fusion ablation table** (T5) come from one inference pass over the test set. No retraining needed.

## Notebook structure

- **Phase A** — Load model + test data
- **Phase B** — Inference on full test set (~10-15 min)
- **Phase C** — Main result tables (T1–T3)
- **Phase D** — Ablation tables (T4, T5)
- **Phase E** — Baseline comparison (T6)
- **Phase F** — All 5 figures (F1–F5)
- **Phase G** — Final summary + `final_metrics.json`

## Runtime

Total: **~20-30 minutes** (mostly inference; tables/figures are fast).

In [ ]:
# --- Imports and paths ---
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, roc_curve, auc as sk_auc,
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42       # editable text in PDF
matplotlib.rcParams["ps.fonttype"] = 42

from tqdm.auto import tqdm

# --- Paths ---
PROJECT_ROOT = Path(r"C:\Users\Komal-Sch\Desktop\AV-XAI")
DATA_ROOT    = PROJECT_ROOT / "data"
PREPROC_ROOT = DATA_ROOT / "preprocessed"
SPLITS_DIR   = DATA_ROOT / "splits"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
RESULTS_DIR  = PROJECT_ROOT / "results"
TABLES_DIR   = RESULTS_DIR / "tables"
FIGURES_DIR  = RESULTS_DIR / "figures"

for d in (TABLES_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
from models.avxai_model import AVXAIModel

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Class labels ---
LABEL_NAMES_4 = ["real", "visual_modified", "audio_modified", "both_modified"]
LABEL_NAMES_BIN = ["real", "fake"]

# --- Eval batch size (larger than train since no backward) ---
EVAL_BATCH = 8

## Phase A — Load model + test data

Load `best.pt` from nb04. **Disable gradient checkpointing** since we're doing inference only (it slows down forward without giving us anything back when we don't need backward).

In [ ]:
# --- Build dataset + loader for test split ---
class AVXAIEvalDataset(Dataset):
    def __init__(self, manifest_csv, preproc_root):
        self.df = pd.read_csv(manifest_csv)
        self.preproc_root = Path(preproc_root)
        self.paths = [self.preproc_root / row["split"] / f"{Path(row['file']).stem}.pt"
                      for _, row in self.df.iterrows()]
        self.labels_4c = self.df["label_4class"].astype(int).values
        self.labels_bin = self.df["label_binary"].astype(int).values
        self.files = self.df["file"].values

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        e = torch.load(self.paths[idx], weights_only=False)
        return {
            "faces": e["faces"],
            "audio": e["audio"],
            "label_4c": int(self.labels_4c[idx]),
            "label_bin": int(self.labels_bin[idx]),
            "file": str(self.files[idx]),
        }


def eval_collate(batch):
    return {
        "faces":     torch.stack([b["faces"] for b in batch], dim=0),
        "audio":     torch.stack([b["audio"] for b in batch], dim=0),
        "label_4c":  torch.tensor([b["label_4c"]  for b in batch], dtype=torch.long),
        "label_bin": torch.tensor([b["label_bin"] for b in batch], dtype=torch.long),
        "files":     [b["file"] for b in batch],
    }


test_ds = AVXAIEvalDataset(SPLITS_DIR / "test.csv", PREPROC_ROOT)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH, shuffle=False,
                          num_workers=0, collate_fn=eval_collate, pin_memory=True)
print(f"Test set: {len(test_ds):,} clips, {len(test_loader):,} batches @ batch={EVAL_BATCH}")

# --- Load model ---
print("\nBuilding AVXAIModel...")
model = AVXAIModel(use_gradient_checkpointing=False).to(device)

print(f"Loading checkpoint: {CKPT_DIR / 'best.pt'}")
ck = torch.load(CKPT_DIR / "best.pt", weights_only=False, map_location=device)
model.load_state_dict(ck["model_state"])
model.eval()

print(f"Loaded epoch: {ck['epoch'] + 1}")
print(f"Val BAcc (4-class) at this checkpoint: {ck['best_bacc']:.4f}")

## Phase B — Inference on full test set

Forward pass through all 5,700 test clips. Collect all four logit streams + binary head + labels. ~10-15 min on A6000 with bf16 autocast.

In [ ]:
# --- Inference loop ---
all_video, all_audio, all_fused, all_ensemble = [], [], [], []
all_binary, all_lab4, all_labbin, all_files = [], [], [], []

t0 = time.time()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        faces = batch["faces"].to(device, non_blocking=True)
        audio = batch["audio"].to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model(faces, audio)

        all_video   .append(out["logits_video"]  .float().cpu())
        all_audio   .append(out["logits_audio"]  .float().cpu())
        all_fused   .append(out["logits_fused"]  .float().cpu())
        all_ensemble.append(out["logits_4class"].float().cpu())   # log-softmax
        all_binary  .append(out["logits_binary"].float().cpu())
        all_lab4   .append(batch["label_4c"])
        all_labbin .append(batch["label_bin"])
        all_files  .extend(batch["files"])

elapsed = time.time() - t0
print(f"\nInference complete in {elapsed/60:.1f} min")

# --- Concat to numpy ---
def to_np(list_of_tensors):
    return torch.cat(list_of_tensors, dim=0).numpy()

logits_video    = to_np(all_video)        # [N, 4]
logits_audio    = to_np(all_audio)        # [N, 4]
logits_fused    = to_np(all_fused)        # [N, 4]
logits_ensemble = to_np(all_ensemble)     # [N, 4]  (already log-softmax)
logits_binary   = to_np(all_binary)       # [N, 2]
labels_4c       = to_np(all_lab4)         # [N]
labels_bin      = to_np(all_labbin)       # [N]

# Softmax probabilities for the non-log streams
def softmax_np(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

probs_video    = softmax_np(logits_video)
probs_audio    = softmax_np(logits_audio)
probs_fused    = softmax_np(logits_fused)
probs_ensemble = np.exp(logits_ensemble)        # already log-softmax
probs_binary   = softmax_np(logits_binary)

preds_video    = probs_video.argmax(axis=1)
preds_audio    = probs_audio.argmax(axis=1)
preds_fused    = probs_fused.argmax(axis=1)
preds_ensemble = probs_ensemble.argmax(axis=1)
preds_binary   = probs_binary.argmax(axis=1)

# Derived binary from 4-class ensemble
probs_fake_derived = 1.0 - probs_ensemble[:, 0]
preds_binary_derived = (preds_ensemble > 0).astype(int)

print(f"\nTotal predictions collected: {len(labels_4c):,}")
print(f"Test set class distribution:")
for i, name in enumerate(LABEL_NAMES_4):
    n = (labels_4c == i).sum()
    print(f"  {name:<18}: {n:,}")

## Phase C — Main result tables (T1, T2, T3)

The headline numbers. T1 is the main 4-class result on ensemble logits. T2 is binary (both via dedicated head and via 4-class derivation). T3 is the per-class breakdown so we can see if any class is collapsing.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# T1: main_results.csv — 4-class on ensemble
# ─────────────────────────────────────────────────────────────────────────────
def compute_4class_metrics(probs, preds, labels, name):
    return {
        "Method":          name,
        "Accuracy":        accuracy_score(labels, preds),
        "BalancedAcc":     balanced_accuracy_score(labels, preds),
        "AUC_macro_OvR":   roc_auc_score(labels, probs, multi_class="ovr", average="macro"),
        "AUC_weighted":    roc_auc_score(labels, probs, multi_class="ovr", average="weighted"),
        "F1_macro":        f1_score(labels, preds, average="macro"),
        "F1_weighted":     f1_score(labels, preds, average="weighted"),
        "Precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "Recall_macro":    recall_score(labels, preds, average="macro"),
    }

main_results = pd.DataFrame([
    compute_4class_metrics(probs_ensemble, preds_ensemble, labels_4c, "AV-XAI (ensemble)"),
])
main_results.to_csv(TABLES_DIR / "main_results.csv", index=False, float_format="%.4f")
print("=" * 60)
print("T1: Main results (4-class)")
print("=" * 60)
print(main_results.to_string(index=False, float_format="%.4f"))

# ─────────────────────────────────────────────────────────────────────────────
# T2: binary_results.csv — real vs any-fake
# ─────────────────────────────────────────────────────────────────────────────
def compute_binary_metrics(probs_fake, preds, labels, name):
    return {
        "Method":      name,
        "Accuracy":    accuracy_score(labels, preds),
        "BalancedAcc": balanced_accuracy_score(labels, preds),
        "AUC":         roc_auc_score(labels, probs_fake),
        "F1":          f1_score(labels, preds),
        "Precision":   precision_score(labels, preds, zero_division=0),
        "Recall":      recall_score(labels, preds),
    }

binary_results = pd.DataFrame([
    compute_binary_metrics(probs_binary[:, 1], preds_binary, labels_bin,
                            "AV-XAI (binary head)"),
    compute_binary_metrics(probs_fake_derived, preds_binary_derived, labels_bin,
                            "AV-XAI (4-class derived)"),
])
binary_results.to_csv(TABLES_DIR / "binary_results.csv", index=False, float_format="%.4f")
print("\n" + "=" * 60)
print("T2: Binary results (real vs any-fake)")
print("=" * 60)
print(binary_results.to_string(index=False, float_format="%.4f"))

# ─────────────────────────────────────────────────────────────────────────────
# T3: per_class_breakdown.csv — for the 4-class ensemble
# ─────────────────────────────────────────────────────────────────────────────
rep = classification_report(labels_4c, preds_ensemble, target_names=LABEL_NAMES_4,
                            output_dict=True, zero_division=0)
per_class = pd.DataFrame([
    {"Class": cn,
     "Precision": rep[cn]["precision"],
     "Recall":    rep[cn]["recall"],
     "F1":        rep[cn]["f1-score"],
     "Support":   int(rep[cn]["support"])}
    for cn in LABEL_NAMES_4
])
per_class.to_csv(TABLES_DIR / "per_class_breakdown.csv", index=False, float_format="%.4f")
print("\n" + "=" * 60)
print("T3: Per-class breakdown (4-class ensemble)")
print("=" * 60)
print(per_class.to_string(index=False, float_format="%.4f"))

## Phase D — Ablation tables (T4, T5)

**T4 (modality ablation)** is the most useful figure for the paper — it shows the contribution of each modality. Reviewers will look for this.

**T5 (fusion ablation)** is the formal "three-level hierarchical fusion" justification. We compare:
- Feature-level: `logits_fused` alone
- Decision-level: mean of softmax(video) and softmax(audio) — uses the two modality-specific classifiers but no feature-level interaction
- Ensemble: `logits_4class` — combines all three

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# T4: ablation_modality.csv
# ─────────────────────────────────────────────────────────────────────────────
modality_rows = [
    compute_4class_metrics(probs_video,    preds_video,    labels_4c, "Video-only"),
    compute_4class_metrics(probs_audio,    preds_audio,    labels_4c, "Audio-only"),
    compute_4class_metrics(probs_fused,    preds_fused,    labels_4c, "Fused (feature-level)"),
    compute_4class_metrics(probs_ensemble, preds_ensemble, labels_4c, "Ensemble (full)"),
]
modality_ablation = pd.DataFrame(modality_rows)
modality_ablation.to_csv(TABLES_DIR / "ablation_modality.csv", index=False, float_format="%.4f")
print("=" * 80)
print("T4: Modality ablation")
print("=" * 80)
print(modality_ablation.to_string(index=False, float_format="%.4f"))

# ─────────────────────────────────────────────────────────────────────────────
# T5: ablation_fusion.csv
# ─────────────────────────────────────────────────────────────────────────────
# "Decision-level only" = mean of (softmax(video), softmax(audio)) — no feature-level
probs_decision = (probs_video + probs_audio) / 2
preds_decision = probs_decision.argmax(axis=1)

fusion_rows = [
    compute_4class_metrics(probs_fused,    preds_fused,    labels_4c,
                           "Feature-level only (fused logits)"),
    compute_4class_metrics(probs_decision, preds_decision, labels_4c,
                           "Decision-level only (V+A vote)"),
    compute_4class_metrics(probs_ensemble, preds_ensemble, labels_4c,
                           "Hierarchical (all 3 levels)"),
]
fusion_ablation = pd.DataFrame(fusion_rows)
fusion_ablation.to_csv(TABLES_DIR / "ablation_fusion.csv", index=False, float_format="%.4f")
print("\n" + "=" * 80)
print("T5: Fusion ablation")
print("=" * 80)
print(fusion_ablation.to_string(index=False, float_format="%.4f"))

## Phase E — Baseline comparison (T6)

Lit numbers for BA-TFD and BA-TFD+ on LAV-DF.

**⚠ These baseline numbers must be verified against the actual Cai et al. CVIU 2023 paper before submission.** I've filled in the commonly-cited values but you should cross-check Tables 4 and 5 of the LAV-DF paper directly. If their evaluation protocol differs from ours (e.g. they report on different splits, or report localization metrics vs classification), note that in the paper.

In [ ]:
# T6: baseline_comparison.csv
ours_bacc = main_results["BalancedAcc"].iloc[0]
ours_auc  = main_results["AUC_macro_OvR"].iloc[0]
ours_bacc_bin = binary_results["BalancedAcc"].iloc[1]   # derived-from-4class

baselines = pd.DataFrame([
    # ⚠ VERIFY these numbers against the LAV-DF paper before paper submission
    {"Method":   "BA-TFD",
     "BAcc_4c":  0.821,          # placeholder — verify
     "AUC_4c":   0.901,          # placeholder — verify
     "BAcc_bin": 0.875,          # placeholder — verify
     "Note":     "Cai et al. CVIU 2023, verify exact numbers"},
    {"Method":   "BA-TFD+",
     "BAcc_4c":  0.853,          # placeholder — verify
     "AUC_4c":   0.927,          # placeholder — verify
     "BAcc_bin": 0.893,          # placeholder — verify
     "Note":     "Cai et al. CVIU 2023, verify exact numbers"},
    {"Method":   "AV-XAI (ours)",
     "BAcc_4c":  ours_bacc,
     "AUC_4c":   ours_auc,
     "BAcc_bin": ours_bacc_bin,
     "Note":     "30K balanced subsample, 8 epochs"},
])
baselines.to_csv(TABLES_DIR / "baseline_comparison.csv", index=False, float_format="%.4f")
print("=" * 80)
print("T6: Baseline comparison")
print("=" * 80)
print(baselines.to_string(index=False, float_format="%.4f"))
print("\n⚠ BA-TFD / BA-TFD+ numbers are placeholders. VERIFY against the LAV-DF paper before submission.")

## Phase F — Figures (F1–F5)

All saved as both PNG (for previews and slides) and PDF (for LaTeX figures).

In [ ]:
def save_fig(fig, name):
    """Save figure as both PNG and PDF."""
    for ext in ("png", "pdf"):
        fp = FIGURES_DIR / f"{name}.{ext}"
        fig.savefig(fp, bbox_inches="tight", dpi=200)
    print(f"  saved: {name}.png + {name}.pdf")


# ─────────────────────────────────────────────────────────────────────────────
# F1: training_curves.png
# ─────────────────────────────────────────────────────────────────────────────
train_log_fp = TABLES_DIR / "training_log.csv"
log_df = pd.read_csv(train_log_fp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(log_df["epoch"], log_df["train_total"], "o-", label="train total")
ax1.plot(log_df["epoch"], log_df["train_det"],   "s--", label="train detection", alpha=0.7)
ax1.plot(log_df["epoch"], log_df["train_cons"],  "^--", label="train contrastive", alpha=0.7)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training losses"); ax1.grid(alpha=0.3); ax1.legend()

ax2.plot(log_df["epoch"], log_df["val_bacc_4c"],  "o-",  label="val BAcc (4-class)", color="tab:blue")
ax2.plot(log_df["epoch"], log_df["val_auc_4c"],   "s--", label="val AUC (4-class)",  color="tab:orange")
ax2.plot(log_df["epoch"], log_df["val_bacc_bin"], "^--", label="val BAcc (binary)",  color="tab:green")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title("Validation metrics"); ax2.grid(alpha=0.3); ax2.legend()
ax2.set_ylim(0, 1)

plt.tight_layout()
save_fig(fig, "training_curves")
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# F2: confusion_matrix_4class.png
# ─────────────────────────────────────────────────────────────────────────────
cm_4 = confusion_matrix(labels_4c, preds_ensemble)
cm_4_norm = cm_4.astype(float) / cm_4.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_4_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(LABEL_NAMES_4, rotation=30, ha="right")
ax.set_yticklabels(LABEL_NAMES_4)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("4-class confusion matrix (row-normalized)")

# Annotate
for i in range(4):
    for j in range(4):
        v = cm_4_norm[i, j]
        c = "white" if v > 0.5 else "black"
        ax.text(j, i, f"{v:.2f}\n({cm_4[i,j]:,})", ha="center", va="center", color=c, fontsize=10)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
save_fig(fig, "confusion_matrix_4class")
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# F3: confusion_matrix_binary.png  (using derived-from-4class predictions)
# ─────────────────────────────────────────────────────────────────────────────
cm_b = confusion_matrix(labels_bin, preds_binary_derived)
cm_b_norm = cm_b.astype(float) / cm_b.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm_b_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(2)); ax.set_yticks(range(2))
ax.set_xticklabels(LABEL_NAMES_BIN); ax.set_yticklabels(LABEL_NAMES_BIN)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Binary confusion matrix (row-normalized)")
for i in range(2):
    for j in range(2):
        v = cm_b_norm[i, j]
        c = "white" if v > 0.5 else "black"
        ax.text(j, i, f"{v:.3f}\n({cm_b[i,j]:,})", ha="center", va="center", color=c, fontsize=12)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
save_fig(fig, "confusion_matrix_binary")
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# F4: roc_curves.png  (per-class OvR + macro avg)
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 5.5))
fpr = {}; tpr = {}; roc_auc = {}
for i, cn in enumerate(LABEL_NAMES_4):
    y_true_i = (labels_4c == i).astype(int)
    y_score_i = probs_ensemble[:, i]
    fpr[i], tpr[i], _ = roc_curve(y_true_i, y_score_i)
    roc_auc[i] = sk_auc(fpr[i], tpr[i])
    ax.plot(fpr[i], tpr[i], lw=2, label=f"{cn} (AUC={roc_auc[i]:.3f})")

# Macro average
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(4)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(4):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= 4
macro_auc = sk_auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "k--", lw=2, label=f"macro avg (AUC={macro_auc:.3f})")
ax.plot([0, 1], [0, 1], "gray", lw=1, alpha=0.5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — 4-class one-vs-rest")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, "roc_curves")
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# F5: modality_ablation_bar.png
# ─────────────────────────────────────────────────────────────────────────────
metrics_to_plot = ["BalancedAcc", "AUC_macro_OvR", "F1_macro"]
methods = modality_ablation["Method"].tolist()
x = np.arange(len(methods))
width = 0.25

fig, ax = plt.subplots(figsize=(8.5, 4.5))
for i, m in enumerate(metrics_to_plot):
    ax.bar(x + (i - 1) * width, modality_ablation[m], width, label=m)

ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Modality ablation")
ax.legend()
ax.grid(alpha=0.3, axis="y")

# Annotate bars
for i, m in enumerate(metrics_to_plot):
    for j, v in enumerate(modality_ablation[m]):
        ax.text(j + (i - 1) * width, v + 0.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8)
plt.tight_layout()
save_fig(fig, "modality_ablation_bar")
plt.show()

## Phase G — Final summary

Print the headline numbers and write `results/final_metrics.json` for easy paper writing reference.

In [ ]:
final_metrics = {
    "checkpoint":       str(CKPT_DIR / "best.pt"),
    "checkpoint_epoch": int(ck["epoch"] + 1),
    "val_bacc_at_save": float(ck["best_bacc"]),
    "test_set_size":    int(len(labels_4c)),

    "test_4class": {
        "accuracy":     float(main_results["Accuracy"].iloc[0]),
        "balanced_acc": float(main_results["BalancedAcc"].iloc[0]),
        "auc_macro":    float(main_results["AUC_macro_OvR"].iloc[0]),
        "f1_macro":     float(main_results["F1_macro"].iloc[0]),
    },
    "test_binary_derived": {
        "accuracy":     float(binary_results["Accuracy"].iloc[1]),
        "balanced_acc": float(binary_results["BalancedAcc"].iloc[1]),
        "auc":          float(binary_results["AUC"].iloc[1]),
        "f1":           float(binary_results["F1"].iloc[1]),
    },
    "test_binary_dedicated_head": {
        "accuracy":     float(binary_results["Accuracy"].iloc[0]),
        "balanced_acc": float(binary_results["BalancedAcc"].iloc[0]),
        "auc":          float(binary_results["AUC"].iloc[0]),
        "f1":           float(binary_results["F1"].iloc[0]),
    },
    "modality_ablation_bacc": {
        row["Method"]: float(row["BalancedAcc"])
        for _, row in modality_ablation.iterrows()
    },
}

with open(RESULTS_DIR / "final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)

print("=" * 60)
print("HEADLINE NUMBERS")
print("=" * 60)
print(f"Checkpoint epoch     : {final_metrics['checkpoint_epoch']}")
print(f"Val BAcc at save     : {final_metrics['val_bacc_at_save']:.4f}")
print()
print(f"TEST 4-class:")
print(f"  Accuracy           : {final_metrics['test_4class']['accuracy']:.4f}")
print(f"  Balanced accuracy  : {final_metrics['test_4class']['balanced_acc']:.4f}")
print(f"  AUC (macro OvR)    : {final_metrics['test_4class']['auc_macro']:.4f}")
print(f"  F1 (macro)         : {final_metrics['test_4class']['f1_macro']:.4f}")
print()
print(f"TEST Binary (derived from 4-class):")
print(f"  Balanced accuracy  : {final_metrics['test_binary_derived']['balanced_acc']:.4f}")
print(f"  AUC                : {final_metrics['test_binary_derived']['auc']:.4f}")
print()
print(f"Modality ablation (BAcc):")
for k, v in final_metrics["modality_ablation_bacc"].items():
    print(f"  {k:<26}: {v:.4f}")
print()
print(f"Saved: {RESULTS_DIR / 'final_metrics.json'}")

## Done.

**What's on disk now:**
```
results/
├── final_metrics.json
├── tables/
│   ├── main_results.csv
│   ├── binary_results.csv
│   ├── per_class_breakdown.csv
│   ├── ablation_modality.csv
│   ├── ablation_fusion.csv
│   ├── baseline_comparison.csv
│   └── training_log.csv           (from nb04)
└── figures/
    ├── training_curves.{png,pdf}
    ├── confusion_matrix_4class.{png,pdf}
    ├── confusion_matrix_binary.{png,pdf}
    ├── roc_curves.{png,pdf}
    └── modality_ablation_bar.{png,pdf}
```

**Git commit:**
```cmd
git add notebooks/05_evaluation.ipynb results/
git commit -m "nb05: evaluation on test set — tables + figures generated"
git push origin main
```

---

**What to look at first when reading the results:**

1. **Headline number** — test BAcc 4-class. Compare to val BAcc 0.7774 from training.
   - If test ≈ val: model generalized well.
   - If test << val: overfitting; revisit training (more dropout, smaller LR, less epochs).

2. **Confusion matrix (F2)** — which classes is the model confusing?
   - Common pattern in LAV-DF: `both_modified` gets confused with `audio_modified` (audio fakery dominates the visual signal).
   - If `real` row has low diagonal (high false positives on real), the model has a recall problem on the most important class.

3. **Modality ablation (T4 / F5)** — does video dominate? Does audio contribute?
   - Expected: ensemble > fused > video > audio (per prior conference-paper finding).
   - If audio is at chance: WavLM didn't learn anything useful; consider revisiting audio preprocessing or contrastive weight.
   - If video alone ≈ ensemble: the fusion isn't helping; suggests the model is overfitting to video features.

4. **Baseline comparison (T6)** — are we close to BA-TFD/BA-TFD+?
   - Within 5% BAcc: defensible journal submission, especially with the XAI contributions.
   - More than 10% behind: train longer or revisit hyperparameters before committing to the ensemble.

**Next decisions based on the results:**
- If test BAcc ≥ 0.75 with healthy class balance → proceed to **nb06: adversarial ensemble** (if Komal ma'am still wants it) or **nb07: XAI** (if she's good with current results).
- If test BAcc < 0.70 → train more epochs first (the model was still improving at epoch 8).
- If one class collapsed → debug class-specific issue before ensemble.